# Customer Sentiment Analysis

EDA + model training notebook.

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocess import preprocess_series

sns.set_style("whitegrid")

## 1. Load Data

In [ ]:
df = pd.read_csv("../Dataset/transcripts.csv")
df.head()

## 2. Preprocess Text

In [ ]:
df["clean_text"] = preprocess_series(df["text"])
df.head()

## 3. Exploratory Data Analysis

In [ ]:
sns.countplot(x="airline_sentiment", data=df)
plt.title("Sentiment Class Distribution")
plt.show()

## 4. TF-IDF + Train/Test Split

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer.fit_transform(df["clean_text"])
y = df["airline_sentiment"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 5. Train Models

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

models = {
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"{name}: {accuracy_score(y_test, preds):.4f}")
    print(classification_report(y_test, preds))

## 6. Save Best Model

In [ ]:
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)
best_model = models["Naive Bayes"]  # update based on results above
joblib.dump(best_model, "../models/sentiment_model.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")